In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
t2012 = pd.read_csv("Tabela5-sem_emprego_2012.csv", sep=";", decimal=",", encoding="utf-8-sig")
t2026 = pd.read_csv("Tabela5-sem_emprego_2026.csv", sep=";", decimal=",", encoding="utf-8-sig")

t2012.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Tabela5-sem_emprego_2012.csv'

In [ ]:
t2012 = t2012.rename(columns={
    "Desocupados - homens (2012 T1)": "homens_2012",
    "Desocupados - mulheres (2012 T1)": "mulheres_2012",
})

t2026 = t2026.rename(columns={
    "Desocupados - homens (2026 T1)": "homens_2026",
    "Desocupados - mulheres (2026 T1)": "mulheres_2026",
})

t2026.head()

In [ ]:
comp = t2012.merge(t2026, on=["Sigla", "Código", "Estado"], how="inner")
comp.head()

In [ ]:
path = r"/content/Tabela 1.1.1.xls"

bruto = pd.read_excel(path, engine="xlrd", header=None)

indicador_1 = bruto.iloc[8:41].copy()
indicador_1.columns = [
    "uf_regiao",
    "total",
    "total_branca",
    "total_preta_parda",
    "homem_branca",
    "homem_preta_parda",
    "mulher_branca",
    "mulher_preta_parda",
]

In [ ]:
regioes = ["Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"]

indicador_estados = indicador_1[~indicador_1["uf_regiao"].isin(["Brasil"] + regioes)].copy()
indicador_estados = indicador_estados.rename(columns={"uf_regiao": "Estado"})

indicador_estados.head()

In [ ]:
comp = comp.merge(
    indicador_estados[["Estado", "total"]].rename(columns={"total": "horas_afazeres_domesticos"}),
    on="Estado",
    how="inner",
)
comp.head()

In [ ]:
ordem = comp.sort_values("mulheres_2026")["Estado"]

longo = comp.melt(
    id_vars=["Sigla", "Código", "Estado"],
    value_vars=["mulheres_2012", "mulheres_2026"],
    var_name="Ano",
    value_name="Participacao_mulheres",
)
longo["Ano"] = longo["Ano"].map({"mulheres_2012": "2012 T1", "mulheres_2026": "2026 T1"})

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
sns.barplot(
    data=longo,
    y="Estado",
    x="Participacao_mulheres",
    hue="Ano",
    order=ordem,
    ax=ax,
)
ax.axvline(50, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("Participação das mulheres entre as pessoas desocupadas (%)")
ax.set_ylabel("")
ax.set_title("Desocupação: participação feminina em 2012 T1 e 2026 T1")
ax.legend(title="Ano")
fig.tight_layout()
plt.show()